In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Penjelasan Dataset

Dataset ini berisi informasi tentang calon mahasiswa yang mengajukan beasiswa. Berikut adalah penjelasan mengenai kolom-kolom yang terdapat dalam dataset ini:

- **Nama Siswa**: Nama calon mahasiswa.
- **Status DTKS**: Status kepesertaan dalam Data Terpadu Kesejahteraan Sosial. Diubah menjadi 1 (Terdata) jika terdata dan 0 (Tidak Terdata) jika tidak.
- **Status P3KE**: Status kepesertaan dalam Program Penanganan Pemenuhan Kebutuhan Energi. Diubah menjadi 1 (Ya) jika terdaftar dan 0 (Tidak) jika tidak.
- **Pekerjaan Ayah**: Pekerjaan ayah calon mahasiswa.
- **Pekerjaan Ibu**: Pekerjaan ibu calon mahasiswa.
- **Status Ayah**: Status pekerjaan ayah calon mahasiswa. Diubah menjadi 1 (Bekerja) jika bekerja dan 0 (Tidak Bekerja) jika tidak.
- **Status Ibu**: Status pekerjaan ibu calon mahasiswa. Diubah menjadi 1 (Bekerja) jika bekerja dan 0 (Tidak Bekerja) jika tidak.
- **Jumlah Tanggungan**: Jumlah tanggungan keluarga calon mahasiswa.
- **Kepemilikan Rumah**: Status kepemilikan rumah keluarga calon mahasiswa.
- **Prestasi**: Prestasi akademik atau non-akademik calon mahasiswa.
- **Gaji Gabungan**: Rata-rata gaji gabungan ayah dan ibu calon mahasiswa.
- **Rendah**: Fitur encoding untuk gaji rendah, diubah menjadi 1 jika gaji di bawah dua juta dan 0 jika tidak.
- **Sedang**: Fitur encoding untuk gaji sedang, diubah menjadi 1 jika gaji antara dua hingga empat juta dan 0 jika tidak.
- **Tinggi**: Fitur encoding untuk gaji tinggi, diubah menjadi 1 jika gaji di atas empat juta dan 0 jika tidak.

Harap dicatat bahwa kolom 'Status DTKS', 'Status P3KE', 'Status Ayah', dan 'Status Ibu' telah diubah menjadi format biner di mana 1 menunjukkan "Ya" dan 0 menunjukkan "Tidak". Selain itu, fitur encoding telah diaplikasikan pada kolom 'Rendah', 'Sedang', dan 'Tinggi' untuk menggambarkan range gaji calon mahasiswa.

Dataset ini digunakan untuk memprediksi apakah calon mahasiswa berhak mendapatkan beasiswa atau tidak berdasarkan informasi yang diberikan.

In [ ]:
df = pd.read_csv('/kaggle/input/subclear1/Sub Dataset Clear1.csv')
df

In [ ]:
df.info()

In [ ]:
S_DTKS = df.groupby('Status DTKS').size()
S_DTKS

In [ ]:
PHA = df.groupby('Penghasilan Ayah').size()
PHA

In [ ]:
PHI = df.groupby('Penghasilan Ibu').size()
PHI

### Data Cleansing dan Preparation

In [ ]:
def convert_gaji_to_numeric(gaji_value):
    if gaji_value == 'Tidak Berpenghasilan' or gaji_value == '-':
        return 0
    elif ' - ' in gaji_value:  # Jika ada range gaji
        range_parts = gaji_value.split(' - ')
        lower = convert_to_numeric(range_parts[0])
        upper = convert_to_numeric(range_parts[1])
        return (lower + upper) / 2
    elif '<' in gaji_value:  # Jika ada angka kurang dari
        value = gaji_value.replace('<', '').replace(' ', '')
        return convert_to_numeric(value) / 2
    else:  # Format numerik biasa
        return convert_to_numeric(gaji_value)

# Fungsi untuk mengonversi string numerik menjadi nilai numerik
def convert_to_numeric(value):
    try:
        value = value.replace('Rp. ', '').replace('.', '').replace(' ', '')
        return int(value)
    except ValueError:
        return 0  
# Konversi range gaji ayah dan ibu menjadi nilai numerik
df['Penghasilan Ayah'] = df['Penghasilan Ayah'].apply(convert_gaji_to_numeric)
df['Penghasilan Ibu'] = df['Penghasilan Ibu'].apply(convert_gaji_to_numeric)

In [ ]:
df['gaji_gabungan'] = (df['Penghasilan Ayah'] + df['Penghasilan Ibu'])

# One-hot encoding berdasarkan kategori gaji gabungan
df['rendah'] = df['gaji_gabungan'] < 2_000_000
df['sedang'] = (df['gaji_gabungan'] >= 2_000_000) & (df['gaji_gabungan'] < 4_000_000)
df['tinggi'] = df['gaji_gabungan'] >= 4_000_000


### Def
Klasifikasi gaji dari hasil akumulasi gaji orangtua
Rendah : Gaji kurang dari 2jt
Sedang : Gaji dari 2-4 Jt
Tinggi : Gaji lebih dari 4jt

In [ ]:
beasiswa_counts = df[['rendah', 'sedang', 'tinggi']].sum()

# Membuat plot batang
plt.bar(beasiswa_counts.index, beasiswa_counts.values)
plt.xlabel('Kategori Beasiswa')
plt.ylabel('Jumlah Mahasiswa')
plt.title('Distribusi Kategori Beasiswa')
plt.show()

In [ ]:
df

In [ ]:
Pres = df.groupby('Prestasi').size()
Pres

In [ ]:
df['Prestasi'] = df['Prestasi'].fillna('Tidak Berprestasi').mask(df['Prestasi'].notna(), 'Berprestasi')

In [ ]:
df.drop(['Sumber Listrik', 'Luas Tanah', 'Luas Bangunan'], axis=1, inplace=True)

In [ ]:
df.drop(['Penghasilan Ayah', 'Penghasilan Ibu'], axis=1, inplace=True)

In [ ]:
df

In [ ]:
PKA = df.groupby('Pekerjaan Ayah').size()
PKA

In [ ]:
encoded_df = pd.get_dummies(df, columns=['Pekerjaan Ayah'], prefix='pekerjaan')
encoded_df

In [ ]:
prestasi_counts = df['Prestasi'].value_counts()

# Membuat plot batang
plt.bar(prestasi_counts.index, prestasi_counts.values)
plt.xlabel('Kategori Prestasi')
plt.ylabel('Jumlah Mahasiswa')
plt.title('Distribusi Prestasi Mahasiswa')
plt.show()

In [ ]:
df.info()

In [ ]:
df['Status DTKS'] = df['Status DTKS'].replace({'Terdata': 'Ya', 'Belum Terdata': 'Tidak'})

In [ ]:
df['Status P3KE'] = df['Status P3KE'].replace({'Terdata': 'Ya', 'Belum Terdata': 'Tidak'})

In [ ]:
df['Status P3KE'] = df['Status P3KE'].replace({'Ya': '1', 'Tidak': '0'})
df['Status DTKS'] = df['Status DTKS'].replace({'Ya': '1', 'Tidak': '0'})

In [ ]:
df['Prestasi'] = df['Prestasi'].replace({'Berprestasi': '1', 'Tidak Berprestasi': '0'})

In [ ]:
df

In [ ]:
df['Status Beasiswa'] = df['Status Beasiswa'].fillna('Tidak Berhak')

In [ ]:
df['Status Beasiswa'] = df['Status Beasiswa'].replace({'Berhak': '1', 'Tidak Berhak': '0'})

In [ ]:
df['Status DTKS'] = df['Status DTKS'].astype(int)
df['Status P3KE'] = df['Status P3KE'].astype(int)
df['Status Beasiswa'] = df['Status Beasiswa'].astype(int)


In [ ]:
df['Prestasi'] = df['Prestasi'].astype(int)

In [ ]:
df

### Pemodelan dan Evaluasi menggunakan SVM dan Decision Tree

## SVM

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Pemisahan data menjadi fitur dan label
X = df[['Status DTKS', 'Status P3KE','Prestasi','gaji_gabungan','rendah', 'sedang', 'tinggi']]
y = df['Status Beasiswa']

# Pemisahan data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Membuat model SVM
model_svm = SVC()

# Melatih model SVM pada data latih
model_svm.fit(X_train, y_train)

# Melakukan prediksi pada data uji
predictions_svm = model_svm.predict(X_test)

# Evaluasi model SVM
accuracy_svm = accuracy_score(y_test, predictions_svm)
conf_matrix_svm = confusion_matrix(y_test, predictions_svm)
classification_rep_svm = classification_report(y_test, predictions_svm)

print('Akurasi model SVM:', accuracy_svm)
print('Confusion Matrix SVM:')
print(conf_matrix_svm)
print('Classification Report SVM:')
print(classification_rep_svm)

In [ ]:
# Visualisasi confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_svm, annot=True, cmap='Blues', fmt='g')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - SVM Model')
plt.show()

## Decision Tree

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Pemisahan data menjadi fitur dan label
X = df[['Status DTKS', 'Status P3KE','Prestasi','gaji_gabungan','rendah', 'sedang', 'tinggi']]
y = df['Status Beasiswa']

# Pemisahan data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Membuat model Decision Tree
model_dt = DecisionTreeClassifier()

# Melatih model Decision Tree pada data latih
model_dt.fit(X_train, y_train)

# Melakukan prediksi pada data uji
predictions_dt = model_dt.predict(X_test)

# Evaluasi model Decision Tree
accuracy_dt = accuracy_score(y_test, predictions_dt)
conf_matrix_dt = confusion_matrix(y_test, predictions_dt)
classification_rep_dt = classification_report(y_test, predictions_dt)

print('Akurasi model Decision Tree:', accuracy_dt)
print('Confusion Matrix Decision Tree:')
print(conf_matrix_dt)
print('Classification Report Decision Tree:')
print(classification_rep_dt)

In [ ]:
# Visualisasi confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_dt, annot=True, cmap='Blues', fmt='g')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - SVM Model')
plt.show()

In [ ]:
feature_importance = model_dt.feature_importances_

# Mendapatkan nama fitur dari X_train
feature_names = X_train.columns

# Membuat bar plot untuk menampilkan feature importance
plt.figure(figsize=(10, 6))
plt.barh(np.arange(len(feature_names)), feature_importance, align='center')
plt.yticks(np.arange(len(feature_names)), feature_names)
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Feature Importance - Decision Tree Model')
plt.show()

In [ ]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)


### XGBOOST

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# Inisialisasi model XGBoost
model_xgb = XGBClassifier()

# Definisikan parameter grid yang akan dijelajahi
param_grid = {
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'n_estimators': [100, 200, 300]
}

# Inisialisasi GridSearchCV dengan model XGBoost dan parameter grid
grid_search = GridSearchCV(model_xgb, param_grid, cv=5, scoring='accuracy')

# Melakukan hyperparameter tuning pada data latih
grid_search.fit(X_train, y_train)

# Menampilkan parameter terbaik yang ditemukan
best_params = grid_search.best_params_
print('Best Parameters:', best_params)

# Melakukan prediksi pada data uji dengan model terbaik
best_model = grid_search.best_estimator_
predictions_tuned = best_model.predict(X_test)

# Evaluasi model dengan parameter terbaik
accuracy_tuned = accuracy_score(y_test, predictions_tuned)
print('Akurasi model setelah tuning:', accuracy_tuned)


In [ ]:
df.info()

### Hyperparameter tuning

In [ ]:
# from sklearn.model_selection import GridSearchCV
# from sklearn.svm import SVC

# # Definisikan parameter grid yang akan dijelajahi
# param_grid = {
#     'C': [0.1, 1, 10, 100],
#     'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
#     'gamma': ['scale', 'auto', 0.1, 1, 10]
# }

# # Inisialisasi model SVC
# model_svc = SVC()

# # Inisialisasi GridSearchCV dengan model SVC dan parameter grid
# grid_search = GridSearchCV(model_svc, param_grid, cv=5, scoring='accuracy')

# # Melakukan hyperparameter tuning pada data latih
# grid_search.fit(X_train, y_train)

# # Menampilkan parameter terbaik yang ditemukan
# best_params = grid_search.best_params_
# print('Best Parameters:', best_params)

# # Melakukan prediksi pada data uji dengan model terbaik
# best_model = grid_search.best_estimator_
# predictions_tuned = best_model.predict(X_test)

# # Evaluasi model dengan parameter terbaik
# accuracy_tuned = accuracy_score(y_test, predictions_tuned)
# print('Akurasi model setelah tuning:', accuracy_tuned)


In [ ]:
# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score

# # Pemisahan data menjadi fitur dan label
# X = df[['Status DTKS', 'Status P3KE','Prestasi', 'gaji_gabungan', 'rendah', 'sedang', 'tinggi']]
# y = df['Status Beasiswa']

# # Pemisahan data menjadi data latih dan data uji
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Definisikan daftar hyperparameter yang ingin dituning
# param_grid = {
#     'C': [0.1, 1, 10],
#     'kernel': ['linear', 'rbf', 'poly'],
#     'gamma': [0.1, 1, 10]
# }

# # Membuat model SVM
# model = SVC()

# # Inisiasi GridSearchCV dengan model SVM dan daftar hyperparameter
# grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)

# # Melakukan tuning pada data latih
# grid_search.fit(X_train, y_train)

# # Mendapatkan model terbaik dari hasil tuning
# best_model = grid_search.best_estimator_

# # Melakukan prediksi pada data uji menggunakan model terbaik
# predictions = best_model.predict(X_test)

# # Evaluasi model terbaik
# accuracy = accuracy_score(y_test, predictions)
# print('Akurasi model terbaik:', accuracy)

# # Menampilkan hyperparameter terbaik
# print('Hyperparameter terbaik:', grid_search.best_params_)
